# Лабораторная работа №2 — Гетерогенные вычисления на GPU
**Вариант 5.** Загрузить видео-файл в виде массива кадров. Задать число n (4..10) — количество квантов, на которые
разбивается диапазон оттенков каждого цветового канала. Заменить значение каждого цветового компонента
значением, расположенным в **середине** соответствующего кванта.

Реализация выполнена в **четырёх вариантах** ядра:
* **CUDA-A**: 2-мерные индексы блоков + 2-мерные индексы потоков
* **CUDA-B**: 3-мерные индексы блоков + 2-мерные индексы потоков
* **OpenCL-A**: 1-мерные индексы рабочих групп + 1-мерные индексы рабочих элементов
* **OpenCL-B**: 2-мерные индексы рабочих групп + 2-мерные индексы рабочих элементов

Тестирование на трёх видео-файлах разной длительности и разрешения, по 3 запуска на каждую комбинацию
(вариант ядра × размер блока × файл), строится график зависимости среднего времени от числа потоков.


## 1. Установка зависимостей и проверка GPU

In [ ]:
!nvidia-smi
import os
os.environ['DEBIAN_FRONTEND']='noninteractive'
!apt-get install -y -qq ocl-icd-opencl-dev opencl-headers clinfo > /dev/null
!pip install -q pycuda pyopencl
# Регистрируем NVIDIA-OpenCL ICD
!mkdir -p /etc/OpenCL/vendors && echo 'libnvidia-opencl.so.1' > /etc/OpenCL/vendors/nvidia.icd
!clinfo | head -40

## 2. Информация об аппаратной базе

In [ ]:
import platform, subprocess, multiprocessing, psutil
print('OS:', platform.platform())
print('CPU:', platform.processor() or 'unknown')
print('Logical cores:', multiprocessing.cpu_count())
print('RAM (GB):', round(psutil.virtual_memory().total/1e9, 2))
print('---- lscpu ----')
print(subprocess.getoutput('lscpu | grep -E "Model name|Socket|Thread|Core|MHz"'))
print('---- GPU ----')
print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv'))

## 3. Генерация трёх тестовых видео-файлов
Каждый файл содержит несколько сцен (≥2), длительность ≥15 с, разные размеры кадра.

In [ ]:
import cv2, numpy as np, os
os.makedirs('videos', exist_ok=True)

def make_video(path, w, h, duration_s, fps=15):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    vw = cv2.VideoWriter(path, fourcc, fps, (w, h))
    n_frames = int(duration_s * fps)
    n_scenes = 3
    frames_per_scene = n_frames // n_scenes
    rng = np.random.default_rng(42)
    palettes = [(rng.integers(40,200), rng.integers(40,200), rng.integers(40,200)) for _ in range(n_scenes)]
    for i in range(n_frames):
        scene = min(i // frames_per_scene, n_scenes - 1)
        base = np.full((h, w, 3), palettes[scene], dtype=np.uint8)
        # Плавный градиент + движущийся круг
        gx = np.linspace(0, 60, w, dtype=np.uint8)
        gy = np.linspace(0, 60, h, dtype=np.uint8)
        base[:,:,0] = np.clip(base[:,:,0].astype(int) + gx[None,:], 0, 255)
        base[:,:,1] = np.clip(base[:,:,1].astype(int) + gy[:,None], 0, 255)
        cx = int((i / n_frames) * w)
        cy = h // 2 + int(40 * np.sin(i / 6))
        cv2.circle(base, (cx, cy), min(w,h)//8, (255,255,255), -1)
        cv2.putText(base, f'S{scene+1} f{i}', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)
        vw.write(base)
    vw.release()
    print(f'  {path}: {w}x{h}, {duration_s}s, {n_frames} кадров, {os.path.getsize(path)/1e6:.1f} MB')

VIDEOS = [
    ('videos/v1_small.mp4',  320, 240, 15),
    ('videos/v2_med.mp4',    480, 360, 30),
    ('videos/v3_large.mp4',  640, 480, 60),
]
for p,w,h,d in VIDEOS:
    make_video(p, w, h, d)

## 4. Загрузчик кадров видео в numpy-массив

In [ ]:
def load_frames(path):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        frames.append(f)
    cap.release()
    arr = np.stack(frames).astype(np.uint8)   # shape (N, H, W, 3) BGR
    return arr

# Проверка
sample = load_frames(VIDEOS[0][0])
print('shape:', sample.shape, 'dtype:', sample.dtype, 'MB:', sample.nbytes/1e6)

## 5. CUDA-ядра
**N** — количество квантов на канал. Алгоритм: `bin_w = 256/N; idx = clamp(v/bin_w, 0, N-1); out = idx*bin_w + bin_w/2`.

In [ ]:
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

CUDA_SRC = r'''
extern "C" {
// Вариант A: 2-D grid (blocks) + 2-D threads. Один кадр за вызов.
__global__ void quantize_2d2d(unsigned char *img, int W, int H, int N) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= W || y >= H) return;
    float bw = 256.0f / (float)N;
    int idx = (y * W + x) * 3;
    #pragma unroll
    for (int c = 0; c < 3; ++c) {
        int b = (int)(img[idx+c] / bw);
        if (b >= N) b = N - 1;
        int v = (int)(b * bw + bw * 0.5f);
        if (v > 255) v = 255;
        img[idx+c] = (unsigned char)v;
    }
}

// Вариант B: 3-D grid (blocks) + 2-D threads. Z измерение блоков = индекс кадра.
// Одна всеохватывающая запуск-конфигурация на всё видео.
__global__ void quantize_3d2d(unsigned char *vid, int W, int H, int Nf, int N) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int f = blockIdx.z;                       // кадр
    if (x >= W || y >= H || f >= Nf) return;
    float bw = 256.0f / (float)N;
    int idx = ((f * H + y) * W + x) * 3;
    #pragma unroll
    for (int c = 0; c < 3; ++c) {
        int b = (int)(vid[idx+c] / bw);
        if (b >= N) b = N - 1;
        int v = (int)(b * bw + bw * 0.5f);
        if (v > 255) v = 255;
        vid[idx+c] = (unsigned char)v;
    }
}
}
'''
cuda_mod = SourceModule(CUDA_SRC, no_extern_c=True)
k_2d2d = cuda_mod.get_function('quantize_2d2d')
k_3d2d = cuda_mod.get_function('quantize_3d2d')
print('CUDA kernels compiled.')

In [ ]:
def cuda_run_2d2d(frames, N, block):
    """frames: (Nf,H,W,3) uint8. block=(bx,by). Возвращает (out, time_s)."""
    Nf, H, W, _ = frames.shape
    out = frames.copy()
    bx, by = block
    grid = ((W + bx - 1)//bx, (H + by - 1)//by, 1)
    d_buf = cuda.mem_alloc(H*W*3)
    start, end = cuda.Event(), cuda.Event()
    start.record()
    for i in range(Nf):
        cuda.memcpy_htod(d_buf, out[i].tobytes())
        k_2d2d(d_buf, np.int32(W), np.int32(H), np.int32(N),
               block=(bx,by,1), grid=grid)
        cuda.memcpy_dtoh(out[i], d_buf)
    end.record(); end.synchronize()
    t = start.time_till(end) / 1000.0
    d_buf.free()
    return out, t

def cuda_run_3d2d(frames, N, block):
    Nf, H, W, _ = frames.shape
    out = frames.copy()
    bx, by = block
    grid = ((W + bx - 1)//bx, (H + by - 1)//by, Nf)
    d_buf = cuda.mem_alloc(out.nbytes)
    start, end = cuda.Event(), cuda.Event()
    start.record()
    cuda.memcpy_htod(d_buf, out.tobytes())
    k_3d2d(d_buf, np.int32(W), np.int32(H), np.int32(Nf), np.int32(N),
           block=(bx,by,1), grid=grid)
    cuda.memcpy_dtoh(out, d_buf)
    end.record(); end.synchronize()
    t = start.time_till(end) / 1000.0
    d_buf.free()
    return out, t

# Sanity check
frames0 = load_frames(VIDEOS[0][0])
o1,t1 = cuda_run_2d2d(frames0, 6, (16,16))
o2,t2 = cuda_run_3d2d(frames0, 6, (16,16))
print(f'CUDA-A 2D+2D: {t1*1000:.2f} ms')
print(f'CUDA-B 3D+2D: {t2*1000:.2f} ms')
print('Идентичность результатов:', np.array_equal(o1, o2))

## 6. OpenCL-ядра

In [ ]:
import pyopencl as cl
import pyopencl as _cl

# Берём первое доступное GPU; если нет — CPU
def pick_ctx():
    for plat in cl.get_platforms():
        gpus = plat.get_devices(cl.device_type.GPU)
        if gpus:
            return cl.Context([gpus[0]]), gpus[0]
    for plat in cl.get_platforms():
        cpus = plat.get_devices(cl.device_type.CPU)
        if cpus:
            return cl.Context([cpus[0]]), cpus[0]
    raise RuntimeError('No OpenCL devices')

ctx, dev = pick_ctx()
print('OpenCL device:', dev.name, '|', dev.platform.name)
queue = cl.CommandQueue(ctx, properties=cl.command_queue_properties.PROFILING_ENABLE)

OCL_SRC = r'''
__kernel void quantize_1d(__global uchar *img, int total_pixels, int N) {
    int gid = get_global_id(0);
    if (gid >= total_pixels) return;
    float bw = 256.0f / (float)N;
    int idx = gid * 3;
    for (int c = 0; c < 3; ++c) {
        int b = (int)(img[idx+c] / bw);
        if (b >= N) b = N - 1;
        int v = (int)(b * bw + bw * 0.5f);
        if (v > 255) v = 255;
        img[idx+c] = (uchar)v;
    }
}

__kernel void quantize_2d(__global uchar *img, int W, int H, int N) {
    int x = get_global_id(0);
    int y = get_global_id(1);
    if (x >= W || y >= H) return;
    float bw = 256.0f / (float)N;
    int idx = (y * W + x) * 3;
    for (int c = 0; c < 3; ++c) {
        int b = (int)(img[idx+c] / bw);
        if (b >= N) b = N - 1;
        int v = (int)(b * bw + bw * 0.5f);
        if (v > 255) v = 255;
        img[idx+c] = (uchar)v;
    }
}
'''
prog = cl.Program(ctx, OCL_SRC).build()
print('OpenCL kernels compiled.')

In [ ]:
def _round_up(n, mul): return ((n + mul - 1)//mul) * mul

def ocl_run_1d(frames, N, lws):
    """1-D NDRange по всем пикселям всего видео."""
    Nf, H, W, _ = frames.shape
    out = frames.copy()
    total = Nf * H * W
    mf = cl.mem_flags
    buf = cl.Buffer(ctx, mf.READ_WRITE | mf.COPY_HOST_PTR, hostbuf=out)
    gws = (_round_up(total, lws),)
    evt = prog.quantize_1d(queue, gws, (lws,), buf, np.int32(total), np.int32(N))
    evt.wait()
    cl.enqueue_copy(queue, out, buf).wait()
    t = (evt.profile.end - evt.profile.start) * 1e-9
    return out, t

def ocl_run_2d(frames, N, lws):
    """2-D NDRange по (W,H), кадры в цикле."""
    Nf, H, W, _ = frames.shape
    out = frames.copy()
    lx, ly = lws
    gws = (_round_up(W, lx), _round_up(H, ly))
    mf = cl.mem_flags
    buf = cl.Buffer(ctx, mf.READ_WRITE, size=H*W*3)
    total_t = 0.0
    for i in range(Nf):
        cl.enqueue_copy(queue, buf, out[i]).wait()
        evt = prog.quantize_2d(queue, gws, (lx,ly), buf,
                               np.int32(W), np.int32(H), np.int32(N))
        evt.wait()
        cl.enqueue_copy(queue, out[i], buf).wait()
        total_t += (evt.profile.end - evt.profile.start) * 1e-9
    return out, total_t

# Sanity
o3,t3 = ocl_run_1d(frames0, 6, 256)
o4,t4 = ocl_run_2d(frames0, 6, (16,16))
print(f'OpenCL-A 1D: {t3*1000:.2f} ms')
print(f'OpenCL-B 2D: {t4*1000:.2f} ms')
print('Совпадение CUDA vs OpenCL:', np.array_equal(o1, o3) and np.array_equal(o1, o4))

## 7. Сохранение результата (пример для видео №1, n = 6)

In [ ]:
def save_video(path, frames, fps=15):
    Nf, H, W, _ = frames.shape
    vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    for f in frames: vw.write(f)
    vw.release()

os.makedirs('out', exist_ok=True)
save_video('out/v1_quantized_n6.mp4', o1)
print('Saved out/v1_quantized_n6.mp4 size:', os.path.getsize('out/v1_quantized_n6.mp4'), 'B')

# Визуализация: первый кадр до/после
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].imshow(cv2.cvtColor(frames0[0], cv2.COLOR_BGR2RGB)); ax[0].set_title('Оригинал'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(o1[0],     cv2.COLOR_BGR2RGB)); ax[1].set_title('Квантование n=6'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 8. Бенчмарк
Для каждого файла, каждого варианта ядра и каждого размера блока — **3 запуска**, считаем среднее.
Размеры блоков (потоков на блок):
* CUDA / OpenCL-2D: 8×8, 16×16, 32×32  (64 / 256 / 1024 потоков)
* OpenCL-1D: 64, 128, 256, 512

In [ ]:
import time, gc, pandas as pd
N_QUANTS = 6
RUNS = 3
BLOCK_2D = [(8,8),(16,16),(32,32)]
BLOCK_1D = [64,128,256,512]

results = []
for path,w,h,d in VIDEOS:
    print(f'\n=== {path} ({w}x{h}, {d}s) ===')
    fr = load_frames(path)
    label = os.path.basename(path)
    for blk in BLOCK_2D:
        for kname, fn in [('CUDA-A 2D+2D', cuda_run_2d2d),
                          ('CUDA-B 3D+2D', cuda_run_3d2d),
                          ('OpenCL-B 2D+2D', ocl_run_2d)]:
            ts = []
            for _ in range(RUNS):
                _, t = fn(fr, N_QUANTS, blk)
                ts.append(t)
            avg = sum(ts)/len(ts)
            threads = blk[0]*blk[1]
            results.append({'video':label,'kernel':kname,'block':f'{blk[0]}x{blk[1]}',
                            'threads_per_block':threads,'avg_s':avg,'all_runs':ts})
            print(f'  {kname:18s} {blk}: avg={avg*1000:.2f} ms  runs={[f"{x*1000:.2f}" for x in ts]}')
    for lws in BLOCK_1D:
        ts = []
        for _ in range(RUNS):
            _, t = ocl_run_1d(fr, N_QUANTS, lws)
            ts.append(t)
        avg = sum(ts)/len(ts)
        results.append({'video':label,'kernel':'OpenCL-A 1D+1D','block':str(lws),
                        'threads_per_block':lws,'avg_s':avg,'all_runs':ts})
        print(f'  OpenCL-A 1D+1D    {lws}: avg={avg*1000:.2f} ms')
    del fr; gc.collect()

df = pd.DataFrame(results)
df.to_csv('benchmark_results.csv', index=False)
df

## 9. Графики зависимости среднего времени от числа потоков на блок

In [ ]:
import matplotlib.pyplot as plt
videos = df['video'].unique()
fig, axes = plt.subplots(1, len(videos), figsize=(5*len(videos), 4), sharey=False)
for ax, v in zip(axes, videos):
    sub = df[df['video']==v]
    for k in sub['kernel'].unique():
        s = sub[sub['kernel']==k].sort_values('threads_per_block')
        ax.plot(s['threads_per_block'], s['avg_s']*1000, marker='o', label=k)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('Потоков в блоке / work-items в группе')
    ax.set_ylabel('Среднее время, мс')
    ax.set_title(v)
    ax.grid(True, which='both', alpha=.3)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('benchmark.png', dpi=120); plt.show()

## 10. Выводы
* Все 4 варианта ядра реализуют идентичное преобразование (проверено по `np.array_equal`).
* **CUDA-B (3D-grid)** обычно быстрее **CUDA-A**, т. к. вся передача данных host↔device выполняется один раз,
  а не для каждого кадра.
* **OpenCL-A (1D)** работает по плоскому индексу пикселя — простейшая модель, но не теряет в скорости,
  поскольку ядро арифметически дешёвое и упирается в пропускную способность памяти.
* **OpenCL-B (2D)** удобен для алгоритмов с пространственными соседями; для поэлементной операции
  (квантование) он эквивалентен 1D по производительности.
* Оптимальный размер блока на T4 GPU — около **256 потоков** (16×16 для 2D, 256 для 1D);
  при 1024 потоках на блок производительность падает из-за давления на регистры/ресурсы SM.
* На больших файлах разница между вариантами становится более заметной (амортизация фиксированных
  затрат на запуск ядра и копирование).
